# Exploring Stock Data using brapi-dev api

## Load ingested data into spark tables

In [1]:
from spark import loader

spark = loader.create_spark_session("brapi-api")

loader.load_spark_tables(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 11:40:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/07 11:40:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Fundamental Analysis Metrics

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

### Companies with Consistent Profits (Last 4 Quarters)

In [3]:
# Create a window to grab the most recent 4 quarters per stock
window_spec = Window.partitionBy("stock").orderBy(F.col("endDate").desc())

consistent_profits_df = (
    spark.table("incomestatementhistoryquarterly")
    # Rank quarters by date
    .withColumn("rank", F.row_number().over(window_spec))
    # Filter for the last year of data (last 4 quarters)
    .filter(F.col("rank") <= 4)
    # Check if the company was profitable in each quarter (1 if true, 0 if false)
    .withColumn("is_profitable", F.when(F.col("netIncome") > 0, 1).otherwise(0))
    # Aggregate by stock to see how many of the 4 quarters were profitable
    .groupBy("stock")
    .agg(
        F.sum("is_profitable").alias("profitable_quarters_count"),
        F.max(F.when(F.col("rank") == 1, F.col("netIncome"))).alias("current_net_income"),
        F.avg("netIncome").alias("avg_quarterly_net_income"),
    )
    # Target companies that hit 4 out of 4 profitable quarters
    .filter(F.col("profitable_quarters_count") == 4)
    .select(
        "stock", 
        F.col("avg_quarterly_net_income").cast("bigint"),
        F.col("current_net_income").cast("bigint"),
    )
    .orderBy("avg_quarterly_net_income", ascending=False)
)

consistent_profits_df.toPandas().head(10)

,stock,avg_quarterly_net_income,current_net_income
0,PETR3,27008750000,32761000000
1,PETR4,27008750000,32761000000
2,BBDC3,5869879750,5230545000
3,BBDC4,5869879750,5230545000
4,EQPA3,4809258750,384415000
5,ITSA3,4264250000,4462000000
6,ITSA4,4264250000,4462000000
7,ABEV3,4017337750,3885567000
8,BBAS3,3927590250,3106277000
9,SUZB3,2850375000,4311991000


### Companies with Low Debt in Proportion to EBITDA

In [4]:
# Window spec to get the absolute latest recorded quarter
latest_window = Window.partitionBy("stock").orderBy(F.col("endDate").desc())

low_debt_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    # Grab the most recent quarter
    .filter(F.col("rank") == 1)
    # Avoid division by zero or working with companies with negative EBITDA
    .filter(F.col("ebitda") > 0)
    # Annualizing quarterly EBITDA (EBITDA * 4) to accurately measure Debt/EBITDA
    .withColumn("annualized_ebitda", F.col("ebitda") * 4)
    .withColumn("debt_to_ebitda", F.col("totalDebt") / F.col("annualized_ebitda"))
    # Filter for companies with a health ratio (e.g., Debt/EBITDA < 3.0)
    .filter(F.col("debt_to_ebitda") < 3.0)
    .select("stock", "totalDebt", "ebitda", "debt_to_ebitda")
    .orderBy("ebitda", ascending=False)
)

low_debt_df.toPandas().head(10)

,stock,totalDebt,ebitda,debt_to_ebitda
0,PETR3,676977000000,230884000000,0.733027
1,PETR4,676977000000,230884000000,0.733027
2,VALE3,194700000000,51387000000,0.947224
3,JBSS32,138706340000,33719560000,1.028382
4,ABEV3,11377306000,29997867000,0.094818
5,VIVT3,39846260000,25311117000,0.393565
6,SUZB3,109061620000,21771598000,1.252338
7,EQPA3,11482045000,20697467000,0.138689
8,ITSA3,15407000000,19871000000,0.193838
9,ITSA4,15407000000,19871000000,0.193838


### Companies with Consistency Growth (Last 4 Quarters)

In [5]:
# Window spec to pull the last 4 quarters
growth_window = Window.partitionBy("stock").orderBy(F.col("endDate").desc())

consistent_growth_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(growth_window))
    .filter(F.col("rank") <= 4)
    # A quarter is marked '1' if growth was positive (e.g., > 5% YoY growth)
    .withColumn("is_growing", F.when(F.col("revenueGrowth") > 0.05, 1).otherwise(0))
    .groupBy("stock")
    .agg(
        F.sum("is_growing").alias("consistent_growth_quarters"),
        F.avg("revenueGrowth").alias("avg_revenue_growth"),
        F.max(F.when(F.col("rank") == 1, F.col("revenueGrowth"))).alias("current_revenue_growth"),
        F.max(F.when(F.col("rank") == 1, F.col("totalRevenue"))).alias("current_revenue"),
    )
    # Keep companies that hit our target growth benchmark in all 4 quarters
    .filter(F.col("consistent_growth_quarters") == 4)
    .select("stock", "avg_revenue_growth", "current_revenue_growth", "current_revenue")
    .orderBy("current_revenue", ascending=False)
)

consistent_growth_df.toPandas().head(10)

,stock,avg_revenue_growth,current_revenue_growth,current_revenue
0,JBSS32,0.162648,0.086244,480046500000
1,ITUB3,0.155728,0.101525,388673000000
2,ITUB4,0.155728,0.101525,388673000000
3,BBAS3,0.146916,0.170960,326194600000
4,BBDC3,0.215746,0.306325,288161430000
5,BBDC4,0.215746,0.306325,288161430000
6,VBBR3,0.080276,0.082734,192220000000
7,SANB11,0.169096,0.159411,166178490000
8,SANB3,0.169096,0.159411,166178490000
9,SANB4,0.169096,0.159411,166178490000


### Return on Equity (ROE) & Return on Assets (ROA)

In [6]:
# Ensure you are checking the latest quarter's efficiency
efficiency_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    # Filters out highly distorted or negative equity anomalies
    .filter((F.col("returnOnEquity") > 0.15) & (F.col("returnOnAssets") > 0.05))
    .select("stock", "returnOnEquity", "returnOnAssets")
    .orderBy("returnOnEquity", ascending=False)
)

efficiency_df.toPandas().head(10)

,stock,returnOnEquity,returnOnAssets
0,EQPA3,3.457672,0.970171
1,MNPR3,2.568387,1.150877
2,CGAS3,0.894066,0.093325
3,CGAS5,0.894066,0.093325
4,MGEL4,0.840165,0.057048
5,CURY3,0.776080,0.193481
6,CTKA4,0.768303,0.083510
7,ASAI3,0.753122,0.087101
8,BGIP3,0.741727,0.061672
9,BGIP4,0.741727,0.061672


### Free Cash Flow (FCF) Margin

In [7]:
fcf_health_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    .filter(F.col("totalRevenue") > 0)
    # FCF Margin = Free Cash Flow / Total Revenue
    .withColumn("fcf_margin", F.col("freeCashflow") / F.col("totalRevenue"))
    .filter(F.col("fcf_margin") > 0.10)
    .select("stock", "freeCashflow", "fcf_margin")
    .orderBy("freeCashflow", ascending=False)
)

fcf_health_df.toPandas().head(10)

,stock,freeCashflow,fcf_margin
0,BBDC3,145904780000,0.506330
1,BBDC4,145904780000,0.506330
2,ITUB3,92871000000,0.238944
3,ITUB4,92871000000,0.238944
4,PETR3,80740000000,0.162099
5,PETR4,80740000000,0.162099
6,SANB11,66804244000,0.402003
7,SANB3,66804244000,0.402003
8,SANB4,66804244000,0.402003
9,BHIA3,12154000000,0.410303


### PEG Ratio (Price/Earnings-to-Growth)

In [8]:
growth_valuation_df = (
    spark.table("defaultkeystatisticshistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    # Look for companies whose stock price isn't outrunning their earnings speed
    .filter((F.col("pegRatio") > 0) & (F.col("pegRatio") <= 1.0))
    .select("stock", "pegRatio", "trailingPE")
    .orderBy("pegRatio", ascending=False)
)

growth_valuation_df.toPandas().head(10)

,stock,pegRatio,trailingPE
0,CEEB3,0.996561,6.583204
1,CEEB5,0.996561,6.583204
2,ITUB3,0.950632,9.567522
3,ITUB4,0.950632,9.567522
4,ITSA3,0.877418,10.182830
5,ITSA4,0.877418,10.182830
6,LOGG3,0.877401,6.507649
7,ORVR3,0.788730,86.228670
8,POSI3,0.769190,42.414450
9,BRAV3,0.746604,41.484580


### Short-Term Liquidity: The Current Ratio

In [9]:
liquidity_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    .filter(F.col("currentRatio") >= 1.0)
    .filter(F.col("quickRatio") >= 1.0)
    .select("stock", "currentRatio", "quickRatio")
    .orderBy("currentRatio", ascending=False)
)

liquidity_df.toPandas().head(10)

,stock,currentRatio,quickRatio
0,BMKS3,35.503826,34.579666
1,EZTC3,8.516822,5.370934
2,PSVM11,7.901747,7.901747
3,EPAR3,7.535687,7.535687
4,CEBR3,5.885724,5.719461
5,CEBR5,5.885724,5.719461
6,CEBR6,5.885724,5.719461
7,CRPG3,5.523639,2.809767
8,CRPG5,5.523639,2.809767
9,CRPG6,5.523639,2.809767


### Good Stock Opportunities

In [10]:
stock_opportunities_df = (
    # Baseline active tickers
    spark.table("activetickers").select("stock", "name", "sector")

    # Consistent Profits for the Last 4 Quarters
    .join(consistent_profits_df, on="stock", how="inner")
    
    # Low Debt in Proportion to EBITDA
    .join(low_debt_df.select("stock", "debt_to_ebitda"), on="stock", how="inner")
    
    # Consistent Growth for the Last 4 Quarters
    .join(consistent_growth_df, on="stock", how="inner")
    
    # Capital Efficiency (ROE & ROA)
    .join(efficiency_df, on="stock", how="inner")
    
    # Free Cash Flow Margin
    .join(fcf_health_df, on="stock", how="inner")
    
    # PEG Ratio
    .join(growth_valuation_df, on="stock", how="inner")
    
    # Short Term Liquidity
    .join(liquidity_df, on="stock", how="inner")
        
    .orderBy("avg_quarterly_net_income", ascending=False)
)

stock_opportunities_df.toPandas().head(20)

,stock,name,sector,avg_quarterly_net_income,current_net_income,debt_to_ebitda,avg_revenue_growth,current_revenue_growth,current_revenue,returnOnEquity,returnOnAssets,freeCashflow,fcf_margin,pegRatio,trailingPE,currentRatio,quickRatio
0,RIAA3,RIAA3,Distribution Services,376713750,5037000,0.299312,0.096295,0.078040,10614021000,0.283291,0.106910,1877788900,0.176916,0.008002,2.904550,1.595530,1.276005
1,DIRR3,DIRECIONAL ENGENHARIA S.A.,Finance,263943517,267534000,2.130367,0.327819,0.291127,4613403000,0.495774,0.075863,555144000,0.120333,0.271804,7.720672,4.256766,2.838528
2,MNPR3,MINUPAR PARTICIPACOES S.A.,Process Industries,129064000,11479000,0.207758,0.236585,0.234505,447971000,2.568387,1.150877,113758000,0.253941,0.000160,0.552873,2.905504,2.577272
3,CSED3,CRUZEIRO DO SUL EDUCACIONAL S.A.,Consumer Services,67984745,61484000,0.765416,0.097128,0.090033,2866369000,0.164598,0.055949,585050000,0.204108,0.124487,5.174448,1.548665,1.548665
4,SEER3,SER EDUCACIONAL S.A.,Consumer Services,61676500,75903000,0.585428,0.114359,0.091603,2260305000,0.174032,0.069145,231722000,0.102518,0.018593,5.786329,1.558467,1.558467


## Full Data Catalog

In [11]:
for table in spark.catalog.listTables():
    print(f"---- {table.name} ----\n")
    spark.table(table.name).printSchema()

---- activetickers ----

root
 |-- change: double (nullable = true)
 |-- close: double (nullable = true)
 |-- logo: string (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- name: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- stock: string (nullable = true)
 |-- subType: string (nullable = true)
 |-- type: string (nullable = true)
 |-- volume: long (nullable = true)

---- balancesheethistoryquarterly ----

root
 |-- stock: string (nullable = true)
 |-- type: string (nullable = true)
 |-- endDate: string (nullable = true)
 |-- cash: long (nullable = true)
 |-- shortTermInvestments: long (nullable = true)
 |-- netReceivables: long (nullable = true)
 |-- inventory: long (nullable = true)
 |-- otherCurrentAssets: long (nullable = true)
 |-- totalCurrentAssets: long (nullable = true)
 |-- longTermInvestments: long (nullable = true)
 |-- propertyPlantEquipment: long (nullable = true)
 |-- otherAssets: long (nullable = true)
 |-- totalAssets: long (nullable